In [9]:
import pandas as pd

df = pd.read_csv("../data/raw/epl_2023_24.csv")

columns_needed = ['Date', 'HomeTeam', 'AwayTeam', 'FTHG', 'FTAG', 'FTR', 'HS', 'AS', 'HST', 'AST', 'HC', 'AC']
df_clean = df[columns_needed]

df_clean['Date'] = pd.to_datetime(df_clean['Date'], dayfirst=True)
df_clean = df_clean.sort_values('Date').reset_index(drop=True)

df_clean['TotalGoals'] = df_clean['FTHG'] + df_clean['FTAG']

df_clean.head()

,Date,HomeTeam,AwayTeam,FTHG,FTAG,FTR,HS,AS,HST,AST,HC,AC,TotalGoals
0,2023-08-11,Burnley,Man City,0,3,A,6,17,1,8,6,5,3
1,2023-08-12,Arsenal,Nott'm Forest,2,1,H,15,6,7,2,8,3,3
2,2023-08-12,Bournemouth,West Ham,1,1,D,14,16,5,3,10,4,2
3,2023-08-12,Brighton,Luton,4,1,H,27,9,12,3,6,7,5
4,2023-08-12,Everton,Fulham,0,1,A,19,9,9,2,10,4,1


In [10]:
import sys
sys.path.append('../src')
from elo import EloEngine

In [11]:
engine = EloEngine()
engine._ratings['TeamA'] = 1600
engine._ratings['TeamB'] = 1500

r_home, r_away, new_home, new_away = engine.update('TeamA', 'TeamB', home_goals=3, away_goals=0)

print(f"Old Elo — Home: {r_home:.1f}, Away: {r_away:.1f}")
print(f"New Elo (3-0 win) — Home: {new_home:.1f}, Away: {new_away:.1f}")

Expected home win probability: 0.64
New Elo â€” Home: 1609.0, Away: 1491.0


In [12]:
# The EloEngine handles initializing default ratings automatically
engine = EloEngine()

In [13]:
home_elo_before = []
away_elo_before = []

for idx, row in df_clean.iterrows():
    home_team = row['HomeTeam']
    away_team = row['AwayTeam']
    home_goals = int(row['FTHG'])
    away_goals = int(row['FTAG'])
    
    # Engine update automatically handles the Margin of Victory (goals) 
    # and Home Advantage, and updates the internal ratings dictionary.
    r_h, r_a, new_h, new_a = engine.update(home_team, away_team, home_goals, away_goals)
    
    home_elo_before.append(r_h)
    away_elo_before.append(r_a)

df_clean['HomeElo'] = home_elo_before
df_clean['AwayElo'] = away_elo_before

print("Engine ratings updated successfully with MoV.")

In [14]:
df_clean[['Date', 'HomeTeam', 'AwayTeam', 'HomeElo', 'AwayElo', 'FTR']].head(10)

,Date,HomeTeam,AwayTeam,HomeElo,AwayElo,FTR
0,2023-08-11,Burnley,Man City,1500.0,1500.0,A
1,2023-08-12,Arsenal,Nott'm Forest,1500.0,1500.0,H
2,2023-08-12,Bournemouth,West Ham,1500.0,1500.0,D
3,2023-08-12,Brighton,Luton,1500.0,1500.0,H
4,2023-08-12,Everton,Fulham,1500.0,1500.0,A
5,2023-08-12,Sheffield United,Crystal Palace,1500.0,1500.0,A
6,2023-08-12,Newcastle,Aston Villa,1500.0,1500.0,H
7,2023-08-13,Brentford,Tottenham,1500.0,1500.0,D
8,2023-08-13,Chelsea,Liverpool,1500.0,1500.0,D
9,2023-08-14,Man United,Wolves,1500.0,1500.0,H


In [15]:
df_clean[df_clean['HomeTeam'] == 'Man City'][['Date', 'HomeTeam', 'AwayTeam', 'HomeElo', 'AwayElo']]

,Date,HomeTeam,AwayTeam,HomeElo,AwayElo
15,2023-08-19,Man City,Newcastle,1512.500000,1512.500000
32,2023-09-02,Man City,Fulham,1535.713422,1500.464457
51,2023-09-23,Man City,Nott'm Forest,1559.059928,1499.152958
87,2023-10-21,Man City,Brighton,1540.880971,1532.261354
104,2023-11-04,Man City,Bournemouth,1564.015724,1446.169211
126,2023-11-25,Man City,Liverpool,1570.037525,1571.204811
139,2023-12-03,Man City,Tottenham,1570.079521,1542.608740
163,2023-12-16,Man City,Crystal Palace,1564.350794,1451.758868
190,2023-12-30,Man City,Sheffield United,1571.181585,1394.184838
215,2024-01-31,Man City,Burnley,1587.217993,1397.158860
